# Business Problem

**Predict whether a bank customer will subscribe to a term deposit based on customer information and previous marketing campaign details.**

**Business Goal**

Identify customers likely to subscribe.
Reduce marketing costs.
Improve campaign success rate.

**Target Variable**

deposit
yes → 1
no → 0

# Import Necessary Libraries

In [ ]:
# Data Manipulation
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Model
from sklearn.linear_model import LogisticRegression

# Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.preprocessing import StandardScaler

# Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

# Cross Validation & Hyperparameter Tuning
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

# Save Model
import joblib

# Ignore Warnings
import warnings
warnings.filterwarnings("ignore")

# Boosting Model
from sklearn.ensemble import GradientBoostingClassifier


: 

# Load Dataset

In [ ]:
bank_data = pd.read_csv("bank.csv")
bank_data

# Data Understanding

In [ ]:
bank_data.head()

In [ ]:
bank_data.info()

In [ ]:
bank_data.describe()

In [ ]:
bank_data.columns

In [ ]:
bank_data.shape

In [ ]:
bank_data["deposit"].value_counts()

# Exploratory Data Analysis (EDA)

## 4.1 Univariate Analysis

### Age Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(bank_data["age"], bins=30, kde=True)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

### Balance Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(bank_data["balance"], bins=30, kde=True)
plt.title("Balance Distribution")
plt.xlabel("Balance")
plt.ylabel("Count")
plt.show()

### Job Count

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(data=bank_data, y="job", order=bank_data["job"].value_counts().index)
plt.title("Job Distribution")
plt.xlabel("Count")
plt.ylabel("Job")
plt.show()

### Education Count

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=bank_data, x="education", order=bank_data["education"].value_counts().index)
plt.title("Education Distribution")
plt.xlabel("Education")
plt.ylabel("Count")
plt.xticks(rotation=20)
plt.show()

### Deposit Count

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=bank_data, x="deposit")
plt.title("Deposit Distribution")
plt.xlabel("Deposit")
plt.ylabel("Count")
plt.show()

## 4.2 Bivariate Analysis

### Age vs Deposit

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=bank_data, x="deposit", y="age")
plt.title("Age vs Deposit")
plt.show()

### Balance vs Deposit

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=bank_data, x="deposit", y="balance")
plt.title("Balance vs Deposit")
plt.show()

### Job vs Deposit

In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(data=bank_data, y="job", hue="deposit")
plt.title("Job vs Deposit")
plt.show()

### Education vs Deposit

In [ ]:
plt.figure(figsize=(10,5))
sns.countplot(data=bank_data, x="education", hue="deposit")
plt.title("Education vs Deposit")
plt.xticks(rotation=20)
plt.show()

### Housing Loan vs Deposit

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=bank_data, x="housing", hue="deposit")
plt.title("Housing Loan vs Deposit")
plt.show()

### Personal Loan vs Deposit

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=bank_data, x="loan", hue="deposit")
plt.title("Personal Loan vs Deposit")
plt.show()

### Marital Status vs Deposit

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(data=bank_data, x="marital", hue="deposit")
plt.title("Marital Status vs Deposit")
plt.show()

### Default vs Deposit

In [ ]:
plt.figure(figsize=(6,5))
sns.countplot(data=bank_data, x="default", hue="deposit")
plt.title("Default vs Deposit")
plt.show()

## 4.3 Multivariate Analysis

### Age vs Balance vs Deposit

In [ ]:
plt.figure(figsize=(8,6))
sns.scatterplot(
    data=bank_data,
    x="age",
    y="balance",
    hue="deposit"
)
plt.title("Age vs Balance vs Deposit")
plt.show()

## Job + Education + Deposit

In [ ]:
plt.figure(figsize=(14,7))
sns.countplot(
    data=bank_data,
    y="job",
    hue="education"
)
plt.title("Job and Education Distribution")
plt.show()

### Job vs Deposit (Faceted by Marital Status)

In [ ]:
g = sns.catplot(
    data=bank_data,
    y="job",
    hue="deposit",
    col="marital",
    kind="count",
    height=5,
    aspect=0.9
)

g.fig.suptitle("Job vs Deposit by Marital Status", y=1.03)
plt.show()

### Correlation Heatmap (Numerical Features)

In [ ]:
plt.figure(figsize=(8,6))

numeric_df = bank_data.select_dtypes(include=["int64", "float64"])

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")
plt.show()

### Pair Plot

In [ ]:
sns.pairplot(
    bank_data,
    vars=["age", "balance"],
    hue="deposit"
)

plt.show()

# Data Cleaning

In [ ]:
bank_data.isnull().sum()

In [ ]:
bank_data.duplicated().sum()

# Feature Selection

**Drop unnecessary columns**

For a realistic prediction model (predict before making a marketing call), drop columns that leak information or are not useful.

In [ ]:
bank_data.drop(
    columns=[
        "contact",
        "day",
        "month",
        "duration",
        "campaign",
        "pdays",
        "previous",
        "poutcome"
    ],
    inplace=True
)

# Feature Engineering

In [ ]:
bank_data["deposit"] = bank_data["deposit"].map({
    "yes": 1,
    "no": 0
})

**Check Categorical Columns**

In [ ]:
bank_data.select_dtypes(include="object").columns

**One-Hot Encoding**

In [ ]:
bank_data = pd.get_dummies(
    bank_data,
    columns=[
        "job",
        "marital",
        "education",
        "default",
        "housing",
        "loan"
    ],
    drop_first=True
)

**Check Transformed Data**

In [ ]:
bank_data.head()

# Define Features and Target

In [ ]:
X = bank_data.drop("deposit", axis=1)
y = bank_data["deposit"]

# Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build Logistic Regression Model

In [ ]:
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Model Prediction

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Model Evaluation

**Accuracy Score**

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy :", accuracy)

**Confusion Matrix**

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

**Confusion Matrix Visualization**

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

**Classification Report**

In [ ]:
print(classification_report(y_test, y_pred))

**ROC-AUC Score**

In [ ]:
roc = roc_auc_score(y_test, y_prob)
print("ROC AUC Score :", roc)

**ROC Curve**

In [ ]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

plt.figure(figsize=(7,5))

plt.plot(fpr, tpr, label="Logistic Regression")
plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

# Cross Validation

In [ ]:
scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores")
print(scores)
print("Average Accuracy :", scores.mean())

# Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

feature_importance = feature_importance.sort_values(
    by="Coefficient",
    ascending=False
)

feature_importance.head(15)

# Save Model

In [ ]:
joblib.dump(model, "bank_deposit_model.pkl")
joblib.dump(scaler, "scaler.pkl")

# ============================================================
# Decision Tree Model
# ============================================================

## 11. Decision Tree Model

Decision Trees **do not require feature scaling**, so we use the original (unscaled) data.

We create a fresh train-test split from the original `X` and `y` to avoid
any contamination from the StandardScaler applied for Logistic Regression.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

# Use original (unscaled) data for Decision Tree
X_train_dt, X_test_dt, y_train_dt, y_test_dt = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train set size:", X_train_dt.shape)
print("Test set size :", X_test_dt.shape)

## 12. Hyperparameter Tuning (GridSearchCV)

We use `GridSearchCV` with 5-fold cross-validation to find the best
combination of hyperparameters for the Decision Tree.

In [ ]:
dt_model = DecisionTreeClassifier(random_state=42)

param_grid = {
    "criterion": ["gini", "entropy"],
    "max_depth": [5, 8, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_dt = GridSearchCV(
    dt_model,
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_dt.fit(X_train_dt, y_train_dt)

print("\nBest Parameters:", grid_dt.best_params_)
print("Best CV Accuracy:", round(grid_dt.best_score_ * 100, 2), "%")

## 13. Prediction

Using the best estimator from GridSearchCV to make predictions on the test set.

In [ ]:
best_dt = grid_dt.best_estimator_

y_pred_dt = best_dt.predict(X_test_dt)
y_prob_dt = best_dt.predict_proba(X_test_dt)[:, 1]

print("Predictions completed.")
print("Sample predictions (first 10):", y_pred_dt[:10])
print("Sample actual     (first 10):", y_test_dt.values[:10])

## 14. Accuracy

In [ ]:
dt_accuracy = accuracy_score(y_test_dt, y_pred_dt)
print("Decision Tree Accuracy:", round(dt_accuracy * 100, 2), "%")

## 15. Classification Report

In [ ]:
print("\nClassification Report:")
print("=" * 55)
print(classification_report(y_test_dt, y_pred_dt, target_names=["No Deposit", "Deposit"]))

## 16. Confusion Matrix

In [ ]:
cm_dt = confusion_matrix(y_test_dt, y_pred_dt)
print("Confusion Matrix:")
print(cm_dt)

**Confusion Matrix Visualization**

In [ ]:
plt.figure(figsize=(6, 5))

sns.heatmap(
    cm_dt,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=["No Deposit", "Deposit"],
    yticklabels=["No Deposit", "Deposit"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Decision Tree - Confusion Matrix")
plt.tight_layout()
plt.show()

**ROC-AUC Score**

In [ ]:
roc_dt = roc_auc_score(y_test_dt, y_prob_dt)
print("Decision Tree ROC AUC Score:", round(roc_dt, 4))

**ROC Curve**

In [ ]:
fpr_dt, tpr_dt, thresholds_dt = roc_curve(y_test_dt, y_prob_dt)

plt.figure(figsize=(7, 5))

plt.plot(fpr_dt, tpr_dt, label="Decision Tree")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Decision Tree - ROC Curve")
plt.legend()
plt.tight_layout()
plt.show()

## 17. Feature Importance

Decision Trees provide a natural measure of feature importance based on
how much each feature contributes to reducing impurity across all splits.

In [ ]:
feature_importance_dt = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_dt.feature_importances_
})

feature_importance_dt = feature_importance_dt.sort_values(
    by="Importance",
    ascending=False
)

print("Top 15 Features:")
print(feature_importance_dt.head(15).to_string(index=False))

In [ ]:
plt.figure(figsize=(10, 8))

# Plot top 15 features
top_features = feature_importance_dt.head(15)

sns.barplot(
    data=top_features,
    x="Importance",
    y="Feature",
    hue="Feature",
    palette="viridis",
    legend=False
)

plt.title("Decision Tree - Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 18. Decision Tree Visualization

Visualizing the tree structure (limited depth for readability).

In [ ]:
plt.figure(figsize=(25, 12))

plot_tree(
    best_dt,
    max_depth=3,
    feature_names=X.columns.tolist(),
    class_names=["No Deposit", "Deposit"],
    filled=True,
    rounded=True,
    fontsize=10
)

plt.title("Decision Tree Visualization (Top 3 Levels)", fontsize=16)
plt.tight_layout()
plt.show()

**Text Representation of the Tree (Top 5 Levels)**

In [ ]:
tree_text = export_text(
    best_dt,
    feature_names=X.columns.tolist(),
    max_depth=5
)

print(tree_text)

## 19. Cross Validation

We evaluate the best Decision Tree model using 5-fold cross-validation
on the full (unscaled) dataset to get a more robust accuracy estimate.

In [ ]:
cv_scores_dt = cross_val_score(
    best_dt,
    X,
    y,
    cv=5,
    scoring="accuracy"
)

print("Cross Validation Scores (Decision Tree)")
print(cv_scores_dt)
print("Average Accuracy:", round(cv_scores_dt.mean() * 100, 2), "%")
print("Std Deviation   :", round(cv_scores_dt.std() * 100, 2), "%")

## 20. Save Decision Tree Model

In [ ]:
joblib.dump(best_dt, "decision_tree_model.pkl")
print("Decision Tree model saved as 'decision_tree_model.pkl'")

# Model Comparison

Compare the performance of Logistic Regression and Decision Tree models.

## Gradient Boosting Classifier

In [ ]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train_dt, y_train_dt)

In [ ]:
y_pred_gb = gb_model.predict(X_test_dt)
y_prob_gb = gb_model.predict_proba(X_test_dt)[:, 1]

gb_accuracy = accuracy_score(y_test_dt, y_pred_gb)
print("Gradient Boosting Accuracy:", round(gb_accuracy * 100, 2), "%")

In [ ]:
print("\nClassification Report:")
print("=" * 55)
print(classification_report(y_test_dt, y_pred_gb, target_names=["No Deposit", "Deposit"]))

roc_gb = roc_auc_score(y_test_dt, y_prob_gb)
print("\nGradient Boosting ROC AUC Score:", round(roc_gb, 4))

In [ ]:
# Logistic Regression accuracy (from earlier cells)
lr_accuracy = accuracy_score(y_test, y_pred)

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Gradient Boosting"],
    "Accuracy (%)": [
        round(lr_accuracy * 100, 2),
        round(dt_accuracy * 100, 2),
        round(gb_accuracy * 100, 2)
    ],
    "ROC-AUC": [
        round(roc_auc_score(y_test, y_prob), 4),
        round(roc_dt, 4),
        round(roc_gb, 4)
    ]
})

print("=" * 55)
print("          MODEL COMPARISON")
print("=" * 55)
print(comparison.to_string(index=False))
print("=" * 55)